# A2780 Staged Growth Estimation

This notebook runs the staged A2780 goal from untreated monoculture through treated coculture, compares mechanistic models, and renders coverage, model ranking, diagnostics, and subgroup overlays. Treated monoculture includes immediate and delayed extinction, Hill-ramp, transit-death, time-decay, and sensitive/tolerant population models.

References: Simeoni et al. 2006 Mathematical Biosciences; Friberg et al. 2002 JCO; oncology TGI time-decay reviews.

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

Cell executed successfully.


In [2]:
using CSV, DataFrames, Plots, Dates, Statistics
include(joinpath(@__DIR__, "..", "src", "MechanicalAutomaticModeling.jl"))
using .MechanicalAutomaticModeling
using GrowthParameterEstimation

Cell executed successfully.


## Density-aware untreated monoculture
The 20k and 30k trajectories are never averaged. Each starts at day 0 with its experiment-design count (67 for 20k and 100 for 30k), while later measured values remain observations. Logistic, Gompertz, and theta-logistic growth are fitted jointly with either shared r/K or symmetric plus-or-minus 5% density contrasts. Fully independent fits are diagnostic and block treatment inheritance when they improve BIC by at least 10.

In [3]:
untreated_out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs("monoculture_untreated"; start = @__DIR__)
for filename in ("monoculture_untreated_pooling_top5.csv", "monoculture_untreated_pooling_status.csv", "monoculture_untreated_pooling_parameter_estimates.csv", "monoculture_untreated_initial_condition_diagnostics.csv")
    path = joinpath(untreated_out.csv, filename)
    isfile(path) && display(CSV.read(path, DataFrame))
end
untreated_grid = joinpath(untreated_out.images, "figures", "monoculture_untreated_pooling_model_grid.png")
isfile(untreated_grid) ? display("image/png", read(untreated_grid)) : println("Untreated pooling grid was not generated.")

Cell executed successfully.


## Run staged workflow

In [4]:
max_time_per_fit = parse(Float64, get(ENV, "A2780_MAX_TIME_PER_FIT", "12.0"))
reuse_outputs = lowercase(get(ENV, "A2780_NOTEBOOK_REUSE_OUTPUTS", "true")) in ("1", "true", "yes")
if reuse_outputs
    staged = MechanicalAutomaticModeling.StagedA2780Workflow.refresh_a2780_output_summary!(start = @__DIR__)
else
    staged = MechanicalAutomaticModeling.StagedA2780Workflow.run_a2780_staged_goal!(start = @__DIR__, max_time_per_fit = max_time_per_fit)
end
CSV.read(staged.overview_path, DataFrame)

4×8 DataFrame
 Row │ condition              status     decoded_rows  fit_rows  best_model    ⋯
     │ String31               String15   Int64         Int64     String31      ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ monoculture_untreated  completed           179        30  theta_logisti ⋯
   2 │ monoculture_treated    completed           504        48  joint_ic_effe
   3 │ coculture_untreated    completed           502         9  lv_asymmetric
   4 │ coculture_treated      completed          1176        21  dual_transit_
                                                               4 columns omitted


## Stage manifest

In [5]:
manifest = CSV.read(staged.manifest_path, DataFrame)
manifest

4×6 DataFrame
 Row │ timestamp_utc        condition              status     message          ⋯
     │ DateTime             String31               String15   String           ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ 2026-08-11T02:03:17  monoculture_untreated  completed  Reused validated ⋯
   2 │ 2026-08-11T02:03:17  monoculture_treated    completed  Reused validated
   3 │ 2026-08-11T02:03:17  coculture_untreated    completed  Reused validated
   4 │ 2026-08-11T02:03:17  coculture_treated      completed  Reused validated
                                                               3 columns omitted


## Coverage by condition

In [6]:
for condition in MechanicalAutomaticModeling.StagedA2780Workflow.STAGED_A2780_CONDITIONS
    out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start = @__DIR__)
    cov_path = joinpath(out.metrics, "$(condition)_a2780_coverage.csv")
    println("\n", condition)
    if isfile(cov_path)
        display(CSV.read(cov_path, DataFrame))
    else
        println("missing coverage: ", cov_path)
    end
end

Cell executed successfully.


## Model rankings

In [7]:
for condition in MechanicalAutomaticModeling.StagedA2780Workflow.STAGED_A2780_CONDITIONS
    out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start = @__DIR__)
    rank_path = joinpath(out.csv, "$(condition)_automatic_model_ranking.csv")
    println("\n", condition)
    if isfile(rank_path)
        rank = CSV.read(rank_path, DataFrame)
        sort_col = :bic in propertynames(rank) ? :bic : (:aic in propertynames(rank) ? :aic : nothing)
        sort_col !== nothing && sort!(rank, sort_col)
        display(first(rank, min(15, nrow(rank))))
    else
        println("missing ranking: ", rank_path)
    end
end

Cell executed successfully.


## Treated monoculture survival and extinction comparison

In [8]:
treated_out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs("monoculture_treated"; start = @__DIR__)
behavior_path = joinpath(treated_out.csv, "monoculture_treated_trajectory_behavior.csv")
joint_path = joinpath(treated_out.csv, "monoculture_treated_joint_dose_model_ranking.csv")
if isfile(behavior_path)
    behavior = CSV.read(behavior_path, DataFrame)
    display(behavior)
end
if isfile(joint_path)
    joint_rank = CSV.read(joint_path, DataFrame)
    sort!(joint_rank, [:cell_line, :bic])
    display(joint_rank)
end
println("IC mapping: 0.67 uM = IC25, 1.0 uM = IC50, 1.47 uM = IC75.")

Cell executed successfully.


### Top five joint-dose models by cell line
For each cell line, BIC is computed from one simultaneous fit to all six trajectories (20k and 30k at IC25/IC50/IC75). Eligible fits compare fully shared treatment effects against a plus-or-minus 5% density contrast in effect amplitude. EC50, Hill shape, onset, decay, transit, clearance, and tolerant fraction remain shared.

In [9]:
cell_line_top5_path = joinpath(treated_out.csv, "monoculture_treated_joint_cell_line_top5.csv")
isfile(cell_line_top5_path) && display(CSV.read(cell_line_top5_path, DataFrame))

Cell executed successfully.


### Best density-coupled model in each treated environment
Each row is a cell-line/starting-density trajectory and each column is IC25, IC50, or IC75. Both density rows for a cell line display the same cell-line-level BIC winner, fitted simultaneously across all six trajectories.

In [10]:
best_environment_figure = joinpath(treated_out.images, "figures", "monoculture_treated_best_joint_model_by_environment.png")
isfile(best_environment_figure) ? display("image/png", read(best_environment_figure)) : println("Best-environment figure was not generated.")

Cell executed successfully.


### Treatment timing architecture audit
The same 12 treated-monoculture trajectories are jointly refitted under five timing structures: independent onset plus gradual activation, shared onset plus gradual activation, onset differences bounded within 0.5 day, resistant gradual-only activation, and resistant onset-only activation. Growth family, inherited r/K, dose response, density pooling, and resistant sensitive/tolerant population structure are otherwise unchanged. Lower BIC is preferred; boundary diagnostics are required because a timing winner can still be poorly identifiable.

In [11]:
timing_ranking_path = joinpath(treated_out.csv, "monoculture_treated_timing_hypothesis_ranking.csv")
timing_parameter_path = joinpath(treated_out.csv, "monoculture_treated_timing_hypothesis_parameters.csv")
timing_identifiability_path = joinpath(treated_out.csv, "monoculture_treated_timing_hypothesis_identifiability.csv")
timing_figure_path = joinpath(treated_out.images, "figures", "monoculture_treated_timing_hypothesis_grid.png")
if isfile(timing_ranking_path)
    timing_ranking = sort!(CSV.read(timing_ranking_path, DataFrame), :bic)
    display(first(timing_ranking, 5))
    timing_winner = String(first(timing_ranking.model))
    if isfile(timing_parameter_path)
        timing_parameters = CSV.read(timing_parameter_path, DataFrame)
        display(timing_parameters[String.(timing_parameters.timing_hypothesis) .== timing_winner, :])
    end
end
isfile(timing_identifiability_path) && display(CSV.read(timing_identifiability_path, DataFrame))
isfile(timing_figure_path) ? display("image/png", read(timing_figure_path)) : println("Timing comparison figure was not generated.")

Cell executed successfully.


### Density coupling and initial-condition control
Each treated trajectory inherits the exact winning untreated growth family and its effective density-specific r/K. Every trajectory starts at day 0 with the experiment-design count: 67 for 20k or 100 for 30k. The first measured point remains an observation and is not substituted for u0. Initial density still changes state evolution through N/K. Residuals are divided by each trajectory's observed peak before optimization so larger-count trajectories do not dominate. Raw SSR is retained for scale-aware diagnostics.

A constant anchored Hill-kill logistic model is autonomous: it can approach a plateau or extinction monotonically, but it cannot rise and then decline. The Naive IC75 transient therefore requires a time-varying model such as Hill-ramp, delayed transit damage, or another delayed-effect mechanism.

In [12]:
u0_diagnostics_path = joinpath(treated_out.csv, "monoculture_treated_joint_initial_condition_diagnostics.csv")
parameter_path = joinpath(treated_out.csv, "monoculture_treated_joint_parameter_estimates.csv")
inheritance_path = joinpath(treated_out.csv, "monoculture_treated_inheritance_audit.csv")
isfile(u0_diagnostics_path) && display(CSV.read(u0_diagnostics_path, DataFrame))
isfile(parameter_path) && display(CSV.read(parameter_path, DataFrame))
isfile(inheritance_path) && display(CSV.read(inheritance_path, DataFrame))

Cell executed successfully.


## BIC plots

In [13]:
for condition in MechanicalAutomaticModeling.StagedA2780Workflow.STAGED_A2780_CONDITIONS
    out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start = @__DIR__)
    rank_path = joinpath(out.csv, "$(condition)_automatic_model_ranking.csv")
    if !isfile(rank_path)
        continue
    end
    rank = CSV.read(rank_path, DataFrame)
    if !(:bic in propertynames(rank)) || nrow(rank) == 0
        continue
    end
    sort!(rank, :bic)
    eligible = :eligible_for_inheritance in propertynames(rank) ? rank[Bool.(rank.eligible_for_inheritance), :] : rank
    top = first(eligible, min(5, nrow(eligible)))
    p = bar(string.(top.model), top.bic; legend = false, xlabel = "Model", ylabel = "BIC", title = "$(condition): top models by BIC", xrotation = 30)
    display(p)
end

Cell executed successfully.


## Coculture whole-system BIC tables and graph tables
Each coculture lineage uses the exact untreated-monoculture growth family, growth rate, carrying capacity, and shape parameter for its cell line and density. Only interaction, death, and treatment parameters are estimated; carrying capacity is not rescaled.

In [14]:
using Base64

function show_png(path)
    encoded = base64encode(read(path))
    name = basename(path)
    display("text/html", "<figure style='margin:0;min-width:0'><img src='data:image/png;base64,$encoded' alt='$name' style='display:block;width:100%;height:auto;border:1px solid #ccd4d6'><figcaption style='font-size:12px;color:#536267;margin-top:6px;overflow-wrap:anywhere'>$name</figcaption></figure>")
end

for condition in ("coculture_untreated", "coculture_treated")
    out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start = @__DIR__)
    println("\n", condition)
    top5_path = joinpath(out.csv, "$(condition)_pooling_top5.csv")
    status_path = joinpath(out.csv, "$(condition)_pooling_status.csv")
    initial_path = joinpath(out.csv, "$(condition)_initial_mix_diagnostics.csv")
    inheritance_name = condition == "coculture_untreated" ? "coculture_untreated_monoculture_inheritance_audit.csv" : "coculture_treated_inheritance_audit.csv"
    coculture_inheritance_path = joinpath(out.csv, inheritance_name)
    nonadditive_path = joinpath(out.csv, "coculture_treated_nonadditive_model_comparison.csv")
    linked_ranking_path = joinpath(out.csv, "linked_treatment_top5.csv")
    linked_status_path = joinpath(out.csv, "linked_treatment_status.csv")
    linked_inheritance_path = joinpath(out.csv, "linked_treatment_effective_parameter_inheritance.csv")
    linked_identifiability_path = joinpath(out.csv, "linked_treatment_identifiability.csv")
    identifiability_path = joinpath(out.csv, "$(condition)_identifiability.csv")
    isfile(top5_path) && display(first(CSV.read(top5_path, DataFrame), 5))
    isfile(status_path) && display(CSV.read(status_path, DataFrame))
    isfile(initial_path) && display(CSV.read(initial_path, DataFrame))
    isfile(coculture_inheritance_path) && display(CSV.read(coculture_inheritance_path, DataFrame))
    condition == "coculture_treated" && isfile(nonadditive_path) && display(CSV.read(nonadditive_path, DataFrame))
    condition == "coculture_treated" && isfile(linked_ranking_path) && display(CSV.read(linked_ranking_path, DataFrame))
    condition == "coculture_treated" && isfile(linked_status_path) && display(CSV.read(linked_status_path, DataFrame))
    condition == "coculture_treated" && isfile(linked_inheritance_path) && display(CSV.read(linked_inheritance_path, DataFrame))
    condition == "coculture_treated" && isfile(linked_identifiability_path) && display(CSV.read(linked_identifiability_path, DataFrame))
    if isfile(identifiability_path) && isfile(status_path)
        status = first(CSV.read(status_path, DataFrame))
        identifiability = CSV.read(identifiability_path, DataFrame)
        display(identifiability[(String.(identifiability.model) .== String(status.winning_model)) .& (String.(identifiability.pooling_mode) .== String(status.winning_pooling_mode)), :])
    end
    figure_path = joinpath(out.images, "figures", "$(condition)_best_mechanistic_fit_grid.png")
    isfile(figure_path) ? show_png(figure_path) : println("Missing combined graph table: ", figure_path)
    nonadditive_figure = joinpath(out.images, "figures", "coculture_treated_nonadditive_simulation_grid.png")
    condition == "coculture_treated" && isfile(nonadditive_figure) && show_png(nonadditive_figure)
    if condition == "coculture_treated"
        for filename in ("linked_treatment_monoculture_grid.png", "linked_treatment_coculture_grid.png", "linked_treatment_hypothesis_comparison_grid.png")
            linked_figure = joinpath(out.images, "figures", filename)
            isfile(linked_figure) && show_png(linked_figure)
        end
    end
end

Cell executed successfully.


## Failure diagnostics

In [15]:
for condition in MechanicalAutomaticModeling.StagedA2780Workflow.STAGED_A2780_CONDITIONS
    out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start = @__DIR__)
    diag = joinpath(out.csv, "diagnostics", "failure_report.csv")
    println("\n", condition)
    overview = CSV.read(staged.overview_path, DataFrame)
    stage_status = String(first(overview[String.(overview.condition) .== condition, :status]))
    if stage_status != "completed" && isfile(diag)
        display(CSV.read(diag, DataFrame))
    else
        println(stage_status == "completed" ? "completed; stale failure files ignored" : "no failure report")
    end
end

Cell executed successfully.


## HTML report

In [16]:
html_path = MechanicalAutomaticModeling.StagedA2780Workflow.render_a2780_report_html(start = @__DIR__)
println(html_path)

Cell executed successfully.
